In [0]:
import requests
from datetime import datetime, timezone
from pyspark.sql import Row

TARGET_TABLE = "finance_intelligence_hub.bronze.exchange_rates"

# -------------------------------------
# Fetch latest rates (base = USD)
# API returns: 1 USD = X of each currency
# So to convert FROM a currency TO USD, we need 1 / rate
# -------------------------------------
def fetch_usd_rates():
    url = "https://open.er-api.com/v6/latest/USD"
    resp = requests.get(url, timeout=15)
    resp.raise_for_status()
    data = resp.json()

    if data.get("result") != "success":
        raise ValueError(f"API did not return success: {data}")

    rates = data["rates"]  # e.g. {"EUR": 0.92, "EGP": 48.5, "GBP": 0.79, ...}
    fetched_at = datetime.now(timezone.utc)
    last_update_utc = data.get("time_last_update_utc")

    rows = []
    for currency, usd_to_currency in rates.items():
        if usd_to_currency == 0:
            continue
        rows.append(
            Row(
                currency=currency,
                usd_to_currency_rate=float(usd_to_currency),      # 1 USD -> X currency
                currency_to_usd_rate=float(1 / usd_to_currency),  # 1 currency -> X USD
                fetched_at=fetched_at,
                source_last_update=last_update_utc,
            )
        )
    return rows

rows = fetch_usd_rates()
print(f"Fetched {len(rows)} currency rates")

rates_df = spark.createDataFrame(rows)

# -------------------------------------
# Overwrite each run — this is a point-in-time snapshot,
# not something you incrementally merge
# -------------------------------------
(
    rates_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET_TABLE)
)

print(f"Saved to {TARGET_TABLE}")